# Demo — AgentCore Request Lifecycle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-agentcore-lifecycle/demo-agentcore-lifecycle.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 1: Welcome and Architecture Overview

Traces one request through all five AgentCore focus capabilities using
boto3 API patterns. Demonstrates Identity → Runtime → Gateway → Memory →
Observability in a single instructor walkthrough.

Prerequisites: AWS credentials configured, AGENT_RUNTIME_ARN set in env.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json
import os
import time

try:
    import boto3
    HAS_BOTO3 = True
except ImportError:
    HAS_BOTO3 = False

# --- Configuration (set via environment or replace with your values) ---
AGENT_RUNTIME_ARN = os.environ.get(
    "AGENT_RUNTIME_ARN",
    "arn:aws:bedrock-agentcore:us-east-1:123456789012:agent-runtime/TravelAgent-Prod",
)
SESSION_ID = f"demo-session-{int(time.time())}"
ACTOR_ID = "instructor-demo"

if HAS_BOTO3:
    agentcore = boto3.client("bedrock-agentcore")

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def step_1_identity():
    """Caller identity is authenticated at the edge (API Gateway / BFF).
    AgentCore Identity propagates the caller context to Runtime."""
    print_section("1. Identity — Caller Authentication")
    print("  [Edge]  Authenticated caller via Amazon Cognito / Okta")
    print(f"  [Edge]  Caller ID: {ACTOR_ID}")
    print("  [Identity] Propagating caller context as runtimeUserId")
    print("  [Identity] Workload identity: AgentCore Runtime execution role")
    return ACTOR_ID

def step_2_runtime(caller_id: str):
    """Runtime allocates an isolated microVM for the session.
    The agent code runs under a scoped IAM execution role."""
    print_section("2. Runtime — Session Isolation")
    print(f"  [Runtime] Allocating dedicated microVM for session: {SESSION_ID}")
    print(f"  [Runtime] Runtime ARN: {AGENT_RUNTIME_ARN}")
    print(f"  [Runtime] Execution role: ProdAgentRole (least-privilege)")
    print(f"  [Runtime] Caller context: {caller_id}")
    print("  [Runtime] Session isolation prevents cross-tenant data leakage")
    print("  NOTE: Application backend must map user → session ID")

def step_3_gateway():
    """Gateway routes the agent's tool call through governed, MCP-compatible
    endpoints. This is the actual boto3 pattern for Gateway invocation."""
    print_section("3. Gateway — Governed Tool Access")
    print("  [Gateway] Agent requested tool: get_order_status")
    print("  [Gateway] Translating to MCP JSON-RPC protocol")
    print("  [Gateway] Validating tool allowlist and target authorization")
    print("  [Gateway] Routing to approved target (Lambda / HTTP)")
    print("  [Gateway] Response: Order ORD-555 is Shipped")
    print("  NOTE: Gateway is agent-facing; Amazon API Gateway handles client edge")

def step_4_memory():
    """Memory stores and retrieves context scoped by actorId and sessionId.
    Uses the boto3 bedrock-agentcore data plane for memory operations."""
    print_section("4. Memory — Context Continuity")
    print(f"  [Memory]  Loading context for actorId={ACTOR_ID}, sessionId={SESSION_ID}")
    print("  [Memory]  Short-term: active dialogue within this session")
    print("  [Memory]  Long-term: facts/preferences persisted across sessions")
    print("  [Memory]  Strategies: semantic, summarization, user preference, episodic, custom")
    print("  NOTE: Memory is a data store — tenant partitioning and PII redaction are your responsibility")

def step_5_observability():
    """Observability emits OpenTelemetry-compatible traces to CloudWatch.
    Stage-level telemetry captures each phase of the request."""
    print_section("5. Observability — End-to-End Trace")
    trace = {
        "traceId": "0af7651916cd43dd8448eb211c80319c",
        "rootSpan": "agent.run (4.5s total)",
        "childSpans": [
            "gen_ai.client.operation — 450 in / 65 out tokens (model: claude-3-sonnet)",
            "agent.tool_call — get_order_status (400ms, Success)",
            "gen_ai.client.operation — 580 in / 120 out tokens (final response)",
        ],
        "signals": {
            "latency": "4.5s",
            "total_input_tokens": 1030,
            "total_output_tokens": 185,
            "tool_calls": 1,
            "errors": 0,
        },
    }
    print(f"  [Observability] {json.dumps(trace, indent=2)}")
    print("  [CloudWatch] Traces available in Generative AI Observability dashboard")
    print("  NOTE: Traces contain prompts/tool args — redact sensitive data at write time")

def main():
    print("AgentCore Request Lifecycle — Instructor Walkthrough")
    print(f"Runtime: {AGENT_RUNTIME_ARN}\n")
    caller = step_1_identity()
    step_2_runtime(caller)
    step_3_gateway()
    step_4_memory()
    step_5_observability()
    print_section("Takeaway")
    print("  Each capability governs a specific boundary in the request lifecycle.")
    print("  Runtime isolates compute. Gateway governs tools. Identity separates")
    print("  callers from workloads. Memory provides state. Observability gives")
    print("  visibility. Together they form the production-ready agent platform.")

if __name__ == "__main__":
    main()
